# QM9 and MoleculeNet Molecular Datasets Demo

This notebook demonstrates five molecular property benchmark datasets prepared for Weisfeiler-Leman (WL) GNN expressiveness analysis:

1. **QM9_Properties** (50,000 molecules): Quantum-mechanical properties (HOMO, LUMO, gap, dipole, polarizability)
2. **MoleculeNet_HIV** (41,127 molecules): HIV replication inhibition classification
3. **MoleculeNet_ESOL** (1,128 molecules): Aqueous solubility regression
4. **MoleculeNet_BBBP** (2,039 molecules): Blood-brain barrier penetration classification
5. **MoleculeNet_Tox21** (7,831 molecules): 12 toxicity targets multi-label classification

**Total: 102,125 molecules** with SMILES strings and labeled properties, formatted for WL expressiveness analysis.

## Setup: Install Dependencies

This cell installs the minimal dependencies needed for the notebook. On Colab, pre-installed core packages are skipped to avoid corrupting compiled C extensions.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages pre-installed on Colab — install locally only to match environment
if 'google.colab' not in sys.modules:
    _pip('json5==0.9.14', 'matplotlib==3.10.0', 'pandas==2.2.2')

## Imports

Import required libraries for data loading, JSON processing, and visualization.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

## Data Loading

Define the data loading helper that works both locally and in Colab via GitHub URL.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-85e404-measuring-molecular-property-expressiven/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini_demo_data.json from GitHub URL with local fallback."""
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")

## Load Data

Load the curated demo dataset.

In [ ]:
data = load_data()
print(f"Loaded {data['metadata']['num_datasets']} datasets with {data['metadata']['total_examples']} total examples")

## Configuration

Define tunable parameters. Set to minimal values for quick testing; scale up as needed.

In [ ]:
# Demo configuration: minimal scale for fast testing
MAX_EXAMPLES_PER_DATASET = 3  # Start with 3 examples per dataset
ENABLE_LOGGING = False  # Minimal logging for demo
VERBOSE = True  # Print summaries

## Process Dataset

Extract examples from the loaded data, applying the demo configuration.

In [ ]:
def process_data(data, max_examples=None):
    """Process and optionally limit the dataset.
    
    Args:
        data: Loaded dataset dict with 'datasets' key
        max_examples: Max examples per dataset (None = use all)
    
    Returns:
        List of (dataset_name, examples_list) tuples
    """
    results = []
    for dataset_dict in data["datasets"]:
        dataset_name = dataset_dict["dataset"]
        examples = dataset_dict["examples"]
        
        if max_examples is not None:
            examples = examples[:max_examples]
        
        results.append((dataset_name, examples))
        if VERBOSE:
            print(f"{dataset_name}: {len(examples)} examples")
    
    return results

# Process the data
processed_datasets = process_data(data, max_examples=MAX_EXAMPLES_PER_DATASET)
print(f"\nProcessed {len(processed_datasets)} datasets")

## Inspect Examples

Display sample molecules and their properties from each dataset.

In [ ]:
def inspect_examples(processed_datasets, max_show=2):
    """Display examples from each dataset.
    
    Args:
        processed_datasets: List of (name, examples) tuples
        max_show: Max examples to display per dataset
    """
    for dataset_name, examples in processed_datasets:
        print(f"\n{'='*60}")
        print(f"Dataset: {dataset_name}")
        print(f"Total examples in this batch: {len(examples)}")
        print(f"{'='*60}")
        
        for i, ex in enumerate(examples[:max_show]):
            print(f"\nExample {i}:")
            print(f"  SMILES: {ex['input'][:60]}..." if len(ex['input']) > 60 else f"  SMILES: {ex['input']}")
            print(f"  Properties: {ex['output'][:100]}..." if len(ex['output']) > 100 else f"  Properties: {ex['output']}")
            print(f"  Task Type: {ex['metadata_task_type']}")

inspect_examples(processed_datasets, max_show=1)

## Validate Schema

Verify that examples conform to the exp_sel_data_out schema.

In [ ]:
def validate_example(ex, dataset_name):
    """Validate that an example has required fields.
    
    Required fields: input, output, metadata_smiles, metadata_dataset, metadata_task_type
    """
    required = ["input", "output", "metadata_smiles", "metadata_dataset", "metadata_task_type"]
    missing = [k for k in required if k not in ex]
    
    if missing:
        return False, f"Missing fields: {missing}"
    
    # Validate field types
    if not isinstance(ex["input"], str) or len(ex["input"]) == 0:
        return False, "input must be non-empty string"
    if not isinstance(ex["output"], str) or len(ex["output"]) == 0:
        return False, "output must be non-empty string"
    if ex["metadata_dataset"] != dataset_name:
        return False, f"metadata_dataset mismatch: {ex['metadata_dataset']} != {dataset_name}"
    
    return True, "Valid"

# Run validation
validation_summary = {}
for dataset_name, examples in processed_datasets:
    valid_count = 0
    errors = []
    for i, ex in enumerate(examples):
        is_valid, msg = validate_example(ex, dataset_name)
        if is_valid:
            valid_count += 1
        else:
            errors.append(f"Example {i}: {msg}")
    
    validation_summary[dataset_name] = {
        "total": len(examples),
        "valid": valid_count,
        "errors": errors
    }
    print(f"{dataset_name}: {valid_count}/{len(examples)} valid")
    if errors and VERBOSE:
        for err in errors[:2]:
            print(f"  - {err}")

## Results & Summary

Display key statistics about the datasets and their properties.

In [ ]:
# Build summary statistics
summary_rows = []
for dataset_name, examples in processed_datasets:
    task_type = examples[0]["metadata_task_type"] if examples else "unknown"
    
    # Extract label info
    try:
        label_names = json.loads(examples[0]["metadata_label_names"])
        num_labels = len(label_names)
    except:
        label_names = []
        num_labels = 0
    
    n_classes = examples[0].get("metadata_n_classes", None) if examples else None
    
    summary_rows.append({
        "Dataset": dataset_name,
        "Examples (Demo)": len(examples),
        "Task Type": task_type,
        "Num Labels": num_labels,
        "Classes": n_classes if n_classes else "—"
    })

df_summary = pd.DataFrame(summary_rows)
print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
print(df_summary.to_string(index=False))
print(f"\nTotal examples in demo: {df_summary['Examples (Demo)'].sum()}")
print(f"Total datasets: {len(df_summary)}")

## Visualization

Plot dataset sizes and task types.

In [ ]:
# Visualize dataset distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Examples per dataset
ax = axes[0]
datasets_names = df_summary["Dataset"].values
examples_counts = df_summary["Examples (Demo)"].values
colors = plt.cm.Set3(range(len(datasets_names)))
ax.bar(datasets_names, examples_counts, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel("Number of Examples", fontsize=11, fontweight='bold')
ax.set_xlabel("Dataset", fontsize=11, fontweight='bold')
ax.set_title("Demo Examples per Dataset", fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
for i, (name, count) in enumerate(zip(datasets_names, examples_counts)):
    ax.text(i, count + 0.1, str(count), ha='center', va='bottom', fontweight='bold')

# Chart 2: Task type distribution
ax = axes[1]
task_types = df_summary["Task Type"].value_counts()
ax.pie(task_types.values, labels=task_types.index, autopct='%1.1f%%', startangle=90, colors=colors)
ax.set_title("Task Type Distribution", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'dataset_overview.png'")

## Sample Molecules & Properties

Display detailed information about the first molecule from each dataset.

In [ ]:
print("\n" + "="*80)
print("DETAILED EXAMPLES (First molecule per dataset)")
print("="*80)

for dataset_name, examples in processed_datasets:
    if not examples:
        continue
    
    ex = examples[0]
    print(f"\n📊 {dataset_name}")
    print("-" * 80)
    print(f"SMILES:           {ex['input']}")
    
    # Parse and pretty-print output
    try:
        props = json.loads(ex['output'])
        print(f"Properties:")
        for key, val in props.items():
            if isinstance(val, (int, float)):
                print(f"  • {key}: {val:.4f}" if isinstance(val, float) else f"  • {key}: {val}")
            else:
                print(f"  • {key}: {val}")
    except:
        print(f"Properties:       {ex['output']}")
    
    print(f"Task Type:        {ex['metadata_task_type']}")
    print(f"Row Index:        {ex.get('metadata_row_index', 'N/A')}")

print("\n" + "="*80)